# Notebook 04 — Final Evaluation on the Test Set

### What this notebook does
This is the **only** notebook that touches the test set, and it touches it **once**. Everything that is *chosen*
(the main model, the calibration, the decision thresholds) is chosen on the **validation** set first, then frozen,
then applied to the test set.

| Step | What | Chosen on | Reviewer comment |
|---|---|---|---|
| 1 | Pick the main model (highest validation AUC, feature set B) | validation | #5 |
| 2 | Calibrate probabilities (Platt scaling) | validation | #4 |
| 3 | Choose decision thresholds (Youden; ≥80% sensitivity) | validation | #11 |
| 4 | Test-set results with **95% confidence intervals** | — | #5, #12 |
| 5 | **DeLong tests**: are AUC differences real? | — | #12 |
| 6 | Calibration, ROC / PR curves, confusion matrices (300 DPI) | — | #4, #11 |
| 7 | Sensitivity analyses: all respondents, survey weights | — | #3, #10 |

**Input** — `brfss_model_ready_v2.csv`, `feature_config_v2.json`, models from Notebooks 03 and 03b
**Output** — `outputs/v2_test_*.csv`, `outputs/v2_thresholds.json`, `outputs/v2_test_predictions.csv`,
`models/v2/calibrators_v2.joblib`, `outputs/figures/fig_*_test.png`

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', 'lightgbm', '-q'])

import pandas as pd
import numpy as np
import os, json, time, joblib, itertools
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

from scipy.stats import rankdata, norm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             roc_curve, precision_recall_curve, confusion_matrix)
from sklearn.calibration import calibration_curve
import sklearn
print(f"scikit-learn {sklearn.__version__}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR     = '/content/drive/MyDrive/stroke_prediction'
READY_PATH   = os.path.join(BASE_DIR, 'data', 'processed', 'brfss_model_ready_v2.csv')
CONFIG_PATH  = os.path.join(BASE_DIR, 'outputs', 'feature_config_v2.json')
MODELS_DIR   = os.path.join(BASE_DIR, 'models', 'v2')
FACT_DIR     = os.path.join(MODELS_DIR, 'factorial')
SRC_DIR      = os.path.join(BASE_DIR, 'src')
OUTPUTS_DIR  = os.path.join(BASE_DIR, 'outputs')
FIG_DIR      = os.path.join(OUTPUTS_DIR, 'figures')
SEL_PATH     = os.path.join(OUTPUTS_DIR, 'v2_selected_features.json')
os.makedirs(FIG_DIR, exist_ok=True)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)          # MissForest models from Notebook 03b need src/stroke_utils.py

# ── Run settings ──────────────────────────────────────────────────────────
SAMPLE       = 'primary'
MAIN_SET     = 'B'          # the main model is chosen among feature set B models (no post-stroke features)
SENS_TARGET  = 0.80         # second threshold: the lowest risk cut-off that still catches 80% of stroke cases
N_BOOT       = 1000
SEED         = 42

config = json.load(open(CONFIG_PATH))
df_all = pd.read_csv(READY_PATH)
df     = df_all[df_all['meta_keep_primary']]
val      = df[df['split'] == 'validation'].reset_index(drop=True)
test     = df[df['split'] == 'test'].reset_index(drop=True)
test_all = df_all[df_all['split'] == 'test'].reset_index(drop=True)      # sensitivity analysis only
y_val, y_test = val['stroke'].to_numpy(), test['stroke'].to_numpy()
print(f"validation {len(val):,} (stroke {y_val.mean()*100:.2f}%) | test {len(test):,} (stroke {y_test.mean()*100:.2f}%) | "
      f"test incl. partial interviews {len(test_all):,}")

### Load every trained model

* **Main models** (Notebook 03): 6 classifiers × feature sets A and B.
* **Factorial models** (Notebook 03b, if it has been run): Logistic Regression and LightGBM ×
  {simple, MissForest} imputation × {all, selected} features.

In [ ]:
MODEL_NAMES = ['LogisticRegression', 'DecisionTree', 'RandomForest', 'XGBoost', 'LightGBM', 'Stacking']
models = {}
for fs, m in itertools.product(['A', 'B'], MODEL_NAMES):
    path = os.path.join(MODELS_DIR, f'{SAMPLE}_{fs}_{m}.joblib')
    if os.path.exists(path):
        models[f'{fs} | {m}'] = dict(pipe=joblib.load(path), features=config[f'features_{fs}'],
                                     group='main', fs=fs, model=m)

if os.path.exists(SEL_PATH):
    sel = json.load(open(SEL_PATH))
    base = sel['base_feature_set']
    for imp, fv, m in itertools.product(['simple', 'missforest'], ['all', 'selected'], ['LogisticRegression', 'LightGBM']):
        path = os.path.join(FACT_DIR, f'{SAMPLE}_{base}_{imp}_{fv}_{m}.joblib')
        if os.path.exists(path):
            models[f'{base}-{fv} ({imp}) | {m}'] = dict(
                pipe=joblib.load(path), features=sel['features_S'] if fv == 'selected' else config[f'features_{base}'],
                group='factorial', fs=base, model=m, imp=imp, fv=fv)

MAIN_LABELS = [l for l in models if models[l]['group'] == 'main']
FACT_LABELS = [l for l in models if models[l]['group'] == 'factorial']
print(f"Loaded {len(MAIN_LABELS)} main models and {len(FACT_LABELS)} factorial models")

raw = {}
for label, d in models.items():
    start = time.time()
    sets = [('val', val), ('test', test)] + ([('test_all', test_all)] if d['group'] == 'main' else [])
    raw[label] = {s: d['pipe'].predict_proba(data[d['features']])[:, 1] for s, data in sets}
    print(f"  {label:<45} predicted in {time.time()-start:.0f}s")

### Step 1 — Pick the main model (on validation, before looking at test)

The rule is fixed in advance: **the feature set B model with the highest validation AUC**. Writing the rule down
before seeing test results is what stops us from "shopping" for the best-looking test number.

In [ ]:
val_auc = {l: roc_auc_score(y_val, raw[l]['val']) for l in MAIN_LABELS}
candidates = [l for l in MAIN_LABELS if models[l]['fs'] == MAIN_SET] or MAIN_LABELS
MAIN = max(candidates, key=val_auc.get)
for l in sorted(candidates, key=val_auc.get, reverse=True):
    print(f"  {l:<30} validation AUC {val_auc[l]:.4f}{'   <- main model' if l == MAIN else ''}")

### Step 2 — Calibration (reviewer comment #4)

Because of class weights, the raw scores are **not** real risks — a model trained with stroke cases counted ~22×
says "40% risk" far too often. **Platt scaling** fixes this: a small logistic regression, fitted on the
**validation** set, maps each raw score to a probability that matches how often stroke really happens.

Platt scaling never changes the *order* of people, so **AUC stays exactly the same** — only the probabilities
(and hence the Brier score and the calibration plot) change.

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

calibrators, cal = {}, {}
for label in models:
    c = LogisticRegression(C=1e6, max_iter=1000).fit(logit(raw[label]['val']).reshape(-1, 1), y_val)
    calibrators[label] = c
    cal[label] = {s: c.predict_proba(logit(p).reshape(-1, 1))[:, 1] for s, p in raw[label].items()}
joblib.dump(calibrators, os.path.join(MODELS_DIR, 'calibrators_v2.joblib'))

def cal_slope(y, p):
    return LogisticRegression(C=1e6, max_iter=1000).fit(logit(p).reshape(-1, 1), y).coef_[0][0]

ref_brier = brier_score_loss(y_test, np.full(len(y_test), y_val.mean()))
cal_rows = []
for label in models:
    p_raw, p_cal = raw[label]['test'], cal[label]['test']
    cal_rows.append({'model': label,
                     'brier_raw': brier_score_loss(y_test, p_raw), 'brier_calibrated': brier_score_loss(y_test, p_cal),
                     'mean_predicted_raw': p_raw.mean(), 'mean_predicted_calibrated': p_cal.mean(),
                     'observed_rate': y_test.mean(), 'calibration_slope': cal_slope(y_test, p_cal)})
cal_df = pd.DataFrame(cal_rows).round(4)
cal_df.to_csv(os.path.join(OUTPUTS_DIR, 'v2_test_calibration.csv'), index=False)
print(f"Brier score of a 'no-skill' model that predicts {y_val.mean():.4f} for everyone: {ref_brier:.4f}\n")
print(cal_df.to_string(index=False))
print("\nCalibration slope: 1 = perfect; < 1 = predictions too extreme; > 1 = too timid.")

### Step 3 — Decision thresholds (on validation)

A model gives a probability; a screening tool needs a **yes/no**. Two cut-offs, both chosen on the validation set:

* **Youden** — the cut-off that maximises sensitivity + specificity − 1 (a balanced choice).
* **Sensitivity ≥ 80%** — the highest cut-off that still flags at least 80% of stroke cases on validation
  (a screening choice: miss few cases, accept more false alarms).

In [ ]:
SENS_KEY = f'sens{int(SENS_TARGET * 100)}'

def pick_thresholds(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return {'youden': float(thr[np.argmax(tpr - fpr)]), SENS_KEY: float(thr[np.argmax(tpr >= SENS_TARGET)])}

thresholds = {label: pick_thresholds(y_val, cal[label]['val']) for label in models}
json.dump({'main_model': MAIN, 'calibration': 'platt', 'thresholds': thresholds},
          open(os.path.join(OUTPUTS_DIR, 'v2_thresholds.json'), 'w'), indent=2)
print(f"Main model {MAIN}: Youden threshold {thresholds[MAIN]['youden']:.4f} | "
      f"{SENS_KEY} threshold {thresholds[MAIN][SENS_KEY]:.4f}  (calibrated probabilities)")

### Step 4 — Test-set results with 95% confidence intervals

* **AUC** — confidence interval from the DeLong method (exact formula, no resampling needed).
* **Everything else** (average precision, Brier, sensitivity, specificity, PPV, NPV, F1) — **bootstrap**: draw the
  test set again with replacement 1,000 times, recompute, and take the middle 95%.

PPV (positive predictive value) answers the question a patient asks: *"I was flagged — how likely is it that I
really had a stroke?"* With a 4% base rate, PPV will be low even for a good model; the chapter must say so.

In [ ]:
def delong(y, preds):
    """Fast DeLong (Sun & Xu, 2014). preds: k x n array of scores. Returns AUCs and their k x k covariance."""
    preds = np.atleast_2d(preds)
    pos, neg = preds[:, y == 1], preds[:, y == 0]
    m, n = pos.shape[1], neg.shape[1]
    tx = np.array([rankdata(r) for r in pos])
    ty = np.array([rankdata(r) for r in neg])
    tz = np.array([rankdata(r) for r in np.hstack([pos, neg])])
    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1) / (2 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1 - (tz[:, m:] - ty) / m
    cov = np.atleast_2d(np.cov(v01)) / m + np.atleast_2d(np.cov(v10)) / n
    return aucs, cov

def threshold_metrics(y, p, t):
    pred = p >= t
    tp, fp = np.sum(pred & (y == 1)), np.sum(pred & (y == 0))
    fn, tn = np.sum(~pred & (y == 1)), np.sum(~pred & (y == 0))
    safe = lambda a, b: a / b if b else np.nan
    return {'sensitivity': safe(tp, tp + fn), 'specificity': safe(tn, tn + fp), 'ppv': safe(tp, tp + fp),
            'npv': safe(tn, tn + fn), 'f1': safe(2 * tp, 2 * tp + fp + fn), 'flagged_pct': 100 * pred.mean()}

def all_metrics(y, p_raw, p_cal, thr):
    out = {'ap': average_precision_score(y, p_raw), 'brier': brier_score_loss(y, p_cal)}
    for tname, t in thr.items():
        out.update({f'{k}@{tname}': v for k, v in threshold_metrics(y, p_cal, t).items()})
    return out

point = {l: all_metrics(y_test, raw[l]['test'], cal[l]['test'], thresholds[l]) for l in models}

start = time.time()
rng = np.random.default_rng(SEED)
boot = {l: [] for l in models}
for _ in range(N_BOOT):
    idx = rng.integers(0, len(y_test), len(y_test))
    yb = y_test[idx]
    for l in models:
        boot[l].append(all_metrics(yb, raw[l]['test'][idx], cal[l]['test'][idx], thresholds[l]))
print(f"Bootstrap ({N_BOOT} resamples x {len(models)} models): {(time.time()-start)/60:.1f} min")

res_rows = []
for l in models:
    aucs, cov = delong(y_test, raw[l]['test'])
    se = np.sqrt(cov[0, 0])
    res_rows.append({'model': l, 'metric': 'auc', 'value': aucs[0],
                     'ci_low': aucs[0] - 1.96 * se, 'ci_high': aucs[0] + 1.96 * se})
    b = pd.DataFrame(boot[l])
    for k, v in point[l].items():
        lo_ci, hi_ci = np.nanpercentile(b[k], [2.5, 97.5])
        res_rows.append({'model': l, 'metric': k, 'value': v, 'ci_low': lo_ci, 'ci_high': hi_ci})
results = pd.DataFrame(res_rows)
results.to_csv(os.path.join(OUTPUTS_DIR, 'v2_test_results.csv'), index=False)

def fmt(l, metric, digits=3):
    r = results[(results['model'] == l) & (results['metric'] == metric)].iloc[0]
    return f"{r['value']:.{digits}f} ({r['ci_low']:.{digits}f}–{r['ci_high']:.{digits}f})"

def results_table(labels, tname):
    cols = {'AUC': 'auc', 'Avg precision': 'ap', 'Brier': 'brier', 'Sensitivity': f'sensitivity@{tname}',
            'Specificity': f'specificity@{tname}', 'PPV': f'ppv@{tname}', 'NPV': f'npv@{tname}'}
    return pd.DataFrame({c: [fmt(l, m) for l in labels] for c, m in cols.items()}, index=labels)

for tname in ['youden', SENS_KEY]:
    tab = results_table(MAIN_LABELS, tname)
    tab.to_csv(os.path.join(OUTPUTS_DIR, f'v2_test_table_{tname}.csv'))
    print(f"\nTest set, main models — threshold: {tname}   [value (95% CI)]")
    print(tab.to_string())
print(f"\nStroke prevalence in test set: {y_test.mean():.4f} (= average precision of a random guess)")

### Step 5 — DeLong tests: is one AUC really higher than another? (reviewer comment #12)

Two models scored on the **same** people make correlated errors, so their AUCs can't be compared by just
checking whether the confidence intervals overlap. The **DeLong test** accounts for that pairing and gives a
p-value for "these two AUCs are equal".

Because we run several tests, p-values are **Holm-adjusted** within each family of comparisons
(the more tests you run, the more likely one looks "significant" by luck).

In [ ]:
def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj

def delong_pair(l1, l2):
    aucs, cov = delong(y_test, np.vstack([raw[l1]['test'], raw[l2]['test']]))
    diff = aucs[0] - aucs[1]
    se = np.sqrt(max(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1], 1e-12))
    return diff, diff - 1.96 * se, diff + 1.96 * se, 2 * norm.sf(abs(diff / se))

families = {f'Main model vs other feature set {MAIN_SET} models':
                [(MAIN, l) for l in MAIN_LABELS if models[l]['fs'] == MAIN_SET and l != MAIN],
            'Feature set A vs B (same model)':
                [(f'A | {m}', f'B | {m}') for m in MODEL_NAMES if f'A | {m}' in models and f'B | {m}' in models]}
if FACT_LABELS:
    fam = []
    for m in ['LogisticRegression', 'LightGBM']:
        ref = f'{base}-all (simple) | {m}'
        for other in [f'{base}-selected (simple) | {m}', f'{base}-all (missforest) | {m}',
                      f'{base}-selected (missforest) | {m}']:
            if ref in models and other in models:
                fam.append((other, ref))
    families['Factorial (vs all features, simple imputation)'] = fam

dl_rows = []
for fam_name, pairs in families.items():
    if not pairs:
        continue
    out = [delong_pair(a, b) for a, b in pairs]
    adj = holm([o[3] for o in out])
    for (a, b), (d, lo_ci, hi_ci, p), pa in zip(pairs, out, adj):
        dl_rows.append({'family': fam_name, 'model_1': a, 'model_2': b, 'auc_diff': d, 'ci_low': lo_ci,
                        'ci_high': hi_ci, 'p_value': p, 'p_holm': pa, 'significant_0.05': pa < 0.05})
delong_df = pd.DataFrame(dl_rows)
delong_df.to_csv(os.path.join(OUTPUTS_DIR, 'v2_test_delong.csv'), index=False)
with pd.option_context('display.float_format', '{:.4f}'.format, 'display.width', 200):
    for fam_name, g in delong_df.groupby('family', sort=False):
        print(f"\n{fam_name}")
        print(g.drop(columns='family').to_string(index=False))

In [ ]:
if FACT_LABELS:
    print("2x2 factorial on the TEST set — AUC (95% CI)")
    for m in ['LogisticRegression', 'LightGBM']:
        tab = pd.DataFrame({fv: [fmt(f'{base}-{fv} ({imp}) | {m}', 'auc', 4) if f'{base}-{fv} ({imp}) | {m}' in models else '—'
                                 for imp in ['simple', 'missforest']] for fv in ['all', 'selected']},
                           index=['simple', 'missforest'])
        print(f"\n{m}\n{tab.to_string()}")
    print(f"\nSelected features: {len(sel['features_S'])} of {len(config[f'features_{base}'])}")
else:
    print("Notebook 03b models not found — factorial test results skipped.")

### Step 6 — Figures (300 DPI)

1. ROC and precision–recall curves for the feature set B models
2. Calibration plot: raw scores vs calibrated probabilities
3. Confusion matrices for the main model at both thresholds

In [ ]:
B_LABELS = [l for l in MAIN_LABELS if models[l]['fs'] == MAIN_SET]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for l in B_LABELS:
    fpr, tpr, _ = roc_curve(y_test, raw[l]['test'])
    prec, rec, _ = precision_recall_curve(y_test, raw[l]['test'])
    name = models[l]['model']
    lw = 2.2 if l == MAIN else 1.2
    axes[0].plot(fpr, tpr, lw=lw, label=f"{name} (AUC {roc_auc_score(y_test, raw[l]['test']):.3f})")
    axes[1].plot(rec, prec, lw=lw, label=f"{name} (AP {average_precision_score(y_test, raw[l]['test']):.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8)
axes[1].axhline(y_test.mean(), color='k', ls='--', lw=0.8, label=f'Random ({y_test.mean():.3f})')
axes[0].set(xlabel='False positive rate (1 − specificity)', ylabel='True positive rate (sensitivity)', title='ROC curve')
axes[1].set(xlabel='Recall (sensitivity)', ylabel='Precision (PPV)', title='Precision–recall curve')
for ax in axes:
    ax.legend(fontsize=8)
fig.suptitle(f'Test set (n = {len(y_test):,}), feature set {MAIN_SET}', fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_roc_pr_test.png'), dpi=300)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, kind in zip(axes, ['raw', 'calibrated']):
    top = 0
    for l in B_LABELS:
        p = raw[l]['test'] if kind == 'raw' else cal[l]['test']
        frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=10, strategy='quantile')
        ax.plot(mean_pred, frac_pos, 'o-', ms=3, lw=2.2 if l == MAIN else 1.2, label=models[l]['model'])
        top = max(top, mean_pred.max(), frac_pos.max())
    ax.plot([0, top], [0, top], 'k--', lw=0.8, label='Perfect calibration')
    ax.set(xlabel='Predicted probability (decile mean)', ylabel='Observed stroke rate',
           title=f"{'Raw (class-weighted) scores' if kind == 'raw' else 'After Platt calibration'}")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_calibration_test.png'), dpi=300)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, tname in zip(axes, ['youden', SENS_KEY]):
    cm = confusion_matrix(y_test, cal[MAIN]['test'] >= thresholds[MAIN][tname])
    ax.imshow(cm, cmap='Blues')
    for i, j in itertools.product(range(2), range(2)):
        ax.text(j, i, f"{cm[i, j]:,}\n({100 * cm[i, j] / cm[i].sum():.1f}% of row)", ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=9)
    ax.set_xticks([0, 1], ['Predicted no stroke', 'Predicted stroke'])
    ax.set_yticks([0, 1], ['No stroke', 'Stroke'])
    ax.set_title(f"{tname} threshold ({thresholds[MAIN][tname]:.3f})", fontsize=10)
fig.suptitle(f'Confusion matrices — {MAIN}, test set', fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_confusion_test.png'), dpi=300)
plt.show()

### Step 7 — Sensitivity analyses

**(a) Partial interviews.** The main analysis excludes respondents who skipped most core questions
(`meta_keep_primary`). Do the models still work if we score **everyone** in the test set — and on the excluded
people alone?

**(b) Survey weights.** BRFSS respondents are not a simple random sample; each carries a weight (`meta_weight`)
that says how many US adults they represent. A **weighted AUC** estimates performance in the US adult population
rather than in this sample (reviewer comment #10).

In [ ]:
y_all = test_all['stroke'].to_numpy()
excluded = ~test_all['meta_keep_primary'].to_numpy(bool)
w = test['meta_weight'].to_numpy(float)
w_ok = ~np.isnan(w)

sens_rows = []
for l in MAIN_LABELS:
    row = {'model': l, 'auc_primary_test': roc_auc_score(y_test, raw[l]['test']),
           'auc_all_test': roc_auc_score(y_all, raw[l]['test_all'])}
    ye = y_all[excluded]
    row['auc_excluded_only'] = roc_auc_score(ye, raw[l]['test_all'][excluded]) if 0 < ye.sum() < len(ye) else np.nan
    row['auc_weighted'] = roc_auc_score(y_test[w_ok], raw[l]['test'][w_ok], sample_weight=w[w_ok])
    sens_rows.append(row)
sens_df = pd.DataFrame(sens_rows).round(4)
sens_df.to_csv(os.path.join(OUTPUTS_DIR, 'v2_test_sensitivity.csv'), index=False)
print(f"Test respondents: primary {len(test):,} | all {len(test_all):,} | excluded (partial interviews) "
      f"{excluded.sum():,}, stroke {y_all[excluded].mean()*100:.2f}%")
print(f"Weighted stroke prevalence (US adults): {np.average(y_test[w_ok], weights=w[w_ok])*100:.2f}% "
      f"vs unweighted {y_test.mean()*100:.2f}%\n")
print(sens_df.to_string(index=False))

In [ ]:
pred_out = test[['stroke', 'age', 'gender', 'meta_race', 'meta_weight']].copy()
for l in MAIN_LABELS:
    pred_out[f'p_{l}'] = cal[l]['test']
pred_out.to_csv(os.path.join(OUTPUTS_DIR, 'v2_test_predictions.csv'), index=False)

m = MAIN
print("=" * 70)
print("  Notebook 04 complete")
print("=" * 70)
print(f"  Main model          : {m}")
print(f"  Test AUC            : {fmt(m, 'auc')}")
print(f"  Average precision   : {fmt(m, 'ap')}")
print(f"  Brier (calibrated)  : {fmt(m, 'brier', 4)}   (no-skill {ref_brier:.4f})")
for tname in ['youden', SENS_KEY]:
    print(f"  @{tname:<7} threshold : sensitivity {fmt(m, f'sensitivity@{tname}')} | "
          f"specificity {fmt(m, f'specificity@{tname}')} | PPV {fmt(m, f'ppv@{tname}')}")
print(f"  Tables  : outputs/v2_test_*.csv | thresholds: outputs/v2_thresholds.json")
print(f"  Figures : {FIG_DIR}/fig_roc_pr_test.png, fig_calibration_test.png, fig_confusion_test.png")
print("=" * 70)